# 크래프톤 라온-1 미니 정식 출시 정리: 프리뷰와 달라진 점, 추론 단계 설정, Apache 2.0, RTX 5090(NVFP4)·맥(MLX)·llama.cpp 실행법

위에서부터 차례로 실행해 보세요.

### llama.cpp(KRAFTON 포크) 받기와 CUDA 빌드 시작 + GGUF 다운로드

In [ ]:
%%bash
mkdir -p /content/w && cd /content/w
git clone -q --depth 1 -b raon-1-mini https://github.com/krafton-ai/llama.cpp
pip -q install -U "huggingface_hub[hf_xet]"
cd llama.cpp
setsid nohup bash -c 'cmake -B build -DGGML_CUDA=ON -DGGML_CUDA_FA=OFF -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_BUILD_BORINGSSL=ON && cmake --build build --config Release -j2 --target llama-server llama-cli' > /content/w/build.log 2>&1 &
cd /content/w
python -c "from huggingface_hub import snapshot_download; print(snapshot_download('KRAFTON/Raon-1-mini-gguf',allow_patterns=['*IQ4_XS*','*.md'],local_dir='/content/w/gguf'))"
sha256sum /content/w/gguf/raon-1-mini-IQ4_XS.gguf

### 빌드가 끝날 때까지 기다리기 (끝날 때까지 이 셀을 반복 실행, 약 1~1.5시간)

In [ ]:
import subprocess,time
for i in range(9):
    if not subprocess.run("pgrep -f '[c]make --build'",shell=True,capture_output=True,text=True).stdout.strip(): break
    time.sleep(30)
print(subprocess.run("tail -2 /content/w/build.log; ls /content/w/llama.cpp/build/bin | grep '^llama-'",shell=True,capture_output=True,text=True).stdout)

### llama-server 띄우고 한국어 질문 + 메모리 확인

In [ ]:
import subprocess,time,urllib.request,json
subprocess.Popen("cd /content/w/llama.cpp && build/bin/llama-server -m /content/w/gguf/raon-1-mini-IQ4_XS.gguf --alias Raon-1-mini --ctx-size 8192 -ngl 99 --n-cpu-moe 6 --reasoning on --host 127.0.0.1 --port 8000 > /content/w/server.log 2>&1",shell=True,start_new_session=True)
for i in range(30):
    time.sleep(10)
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=3); break
    except Exception: pass
def ask(q,effort=None,max_tokens=1500):
    body={'model':'Raon-1-mini','messages':[{'role':'user','content':q}],'max_tokens':max_tokens,'temperature':1.0,'top_p':0.95,'top_k':20,'presence_penalty':1.5}
    if effort: body['reasoning_effort']=effort
    t=time.time()
    r=json.load(urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8000/v1/chat/completions',json.dumps(body).encode(),{'Content-Type':'application/json'}),timeout=280))
    m=r['choices'][0]['message']
    return {'sec':round(time.time()-t,1),'tokens':r['usage']['completion_tokens'],'finish':r['choices'][0]['finish_reason'],'reasoning_chars':len(m.get('reasoning_content') or ''),'content':m.get('content')}
o=ask('서울에서 부산까지 KTX로 가는 것과 고속버스로 가는 것의 장단점을 세 줄로 비교해 줘.')
print(o)
print(subprocess.run('nvidia-smi --query-gpu=memory.used,memory.total --format=csv; ps -o rss,cmd -C llama-server | cut -c1-60',shell=True,capture_output=True,text=True).stdout)

### 추론 단계 none / low / max 비교

In [ ]:
Q='농부가 17마리 양을 가지고 있었는데 9마리를 제외하고 모두 도망갔습니다. 몇 마리가 남았나요? 그리고 1부터 100까지 소수의 개수를 구하세요.'
for e,mt in [('none',1500),('low',1500),('max',3000)]:
    o=ask(Q,e,mt)
    print('==',e,o['sec'],'s tokens',o['tokens'],o['finish'],'reasoning_chars',o['reasoning_chars'])
    print(o['content'])

### 같은 질문 영어로 보내기

In [ ]:
o=ask('Compare traveling from Seoul to Busan by KTX versus express bus, in three lines.')
print(o)